# CoNLL-2003 BERT baselines

Run this notebook with the same fine-tuning settings as the capitalization-embedding model. The core comparison is `bert-base-uncased` vs `bert-base-cased` vs the custom model from notebook 02.

In [ ]:
from pathlib import Path
import os

COLAB_REPO = Path("/content/drive/MyDrive/Github/CapitalizationEmbeddings")
try:
    from google.colab import drive

    if not COLAB_REPO.exists():
        drive.mount("/content/drive")
except Exception:
    pass

if COLAB_REPO.exists():
    os.chdir(COLAB_REPO)

print("repo:", Path.cwd())
%pip install -q -e . -r requirements-colab.txt

from capitalization_embeddings import configure_huggingface_cache
HF_CACHE_DIR = configure_huggingface_cache()
print("HF cache:", HF_CACHE_DIR)


In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

In [ ]:
MODEL_NAMES = ["bert-base-uncased", "bert-base-cased"]
MAX_LENGTH = 192
from capitalization_embeddings import checkpoint_dir

OUTPUT_ROOT = checkpoint_dir("baselines")

NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 16
LEARNING_RATE = 3e-5

In [ ]:
from datasets import load_dataset

raw = load_dataset("lhoestq/conll2003")
ner_feature = raw["train"].features["ner_tags"].feature
if hasattr(ner_feature, "names"):
    label_list = ner_feature.names
else:
    label_list = [
        "O",
        "B-PER",
        "I-PER",
        "B-ORG",
        "I-ORG",
        "B-LOC",
        "I-LOC",
        "B-MISC",
        "I-MISC",
    ]
id2label = {index: label for index, label in enumerate(label_list)}
label2id = {label: index for index, label in id2label.items()}
print(label_list)

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = [
        [label_list[pred] for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[label] for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

def tokenize_and_align_labels(examples, tokenizer):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []
    for batch_index, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=batch_index)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(word_labels[word_id])
            else:
                label_ids.append(-100)
            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

In [ ]:
import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
)
from capitalization_embeddings import make_trainer, make_training_arguments

def run_baseline(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    tokenized = raw.map(
        lambda examples: tokenize_and_align_labels(examples, tokenizer),
        batched=True,
        remove_columns=raw["train"].column_names,
    )

    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    output_dir = f"{OUTPUT_ROOT}/{model_name.replace('/', '_')}"
    training_args = make_training_arguments(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    trainer = make_trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate(tokenized["test"])
    trainer.save_model(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")
    return metrics

results = {model_name: run_baseline(model_name) for model_name in MODEL_NAMES}
results